# 🚀 Apigee X + Private Cloud Run Persistent JDBC/ODBC Connector — One-Click Deployer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/luiscuellarh/apigee-cloudrun-jdbc-policy/blob/main/colab/deploy_cloudrun_apigee.ipynb)

This interactive **Google Colab / Colab Enterprise** notebook deploys and verifies the **Apigee X + Private Cloud Run Persistent Multi-Datasource JDBC/ODBC Connector & Policy** with a **zero-code, one-click Play button experience**.

![JDBC Architecture Overview](https://raw.githubusercontent.com/luiscuellarh/apigee-cloudrun-jdbc-policy/main/docs/images/jdbc-architecture-overview.svg)

### How to Use This Notebook
1. Enter your **`PROJECT_ID`** and **`REGION`** in **Step 1** (leave `APIGEE_ENV` and `APIGEE_HOSTNAME` blank to auto-discover them via the Apigee Management API).
2. Click the **▶️ Play button** on **Step 1** through **Step 6** in order.
3. **(Optional) Step 7** lets you spin up a **Private Compute Engine PostgreSQL 16 Container VM (`10.x.x.x:5432`)**, attach **Cloud Run Direct VPC Egress**, test private VPC database queries through Apigee, and delete the VM with one click when done.

In [ ]:
#@title Step 1: Configuration & Google Cloud Authentication { display-mode: "form" }
PROJECT_ID = "" #@param {type:"string"}
REGION = "europe-west2" #@param ["europe-west2", "europe-west1", "us-central1", "us-east1", "asia-southeast1"] {allow-input: true}
SERVICE_NAME = "jdbc-policy-connector" #@param {type:"string"}
CUSTOM_AUDIENCE = "jdbc-policy-service" #@param {type:"string"}
APIGEE_ENV = "" #@param {type:"string"}
APIGEE_HOSTNAME = "" #@param {type:"string"}

import os
import sys
import json
import time
import html
import shutil
import subprocess
from datetime import datetime, timezone
from IPython.display import HTML, display

REPO_URL = "https://github.com/luiscuellarh/apigee-cloudrun-jdbc-policy.git"

UI_CSS = """
<style>
.ag-card {
  font-family: 'Google Sans', -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
  border-radius: 12px;
  border: 1px solid #dadce0;
  background: #ffffff;
  color: #202124;
  margin: 12px 0;
  box-shadow: 0 2px 6px rgba(60,64,67,0.08);
  overflow: hidden;
}
.ag-card-header {
  display: flex;
  align-items: center;
  justify-content: space-between;
  padding: 14px 18px;
  border-bottom: 1px solid #e8eaed;
  background: #f8f9fa;
}
.ag-card-title {
  font-size: 15px;
  font-weight: 600;
  display: flex;
  align-items: center;
  gap: 8px;
}
.ag-badge {
  font-size: 11px;
  font-weight: 700;
  padding: 4px 10px;
  border-radius: 999px;
  letter-spacing: 0.3px;
  text-transform: uppercase;
}
.ag-badge-success { background: #e6f4ea; color: #137333; border: 1px solid #ceead6; }
.ag-badge-error   { background: #fce8e6; color: #c5221f; border: 1px solid #fad2cf; }
.ag-badge-running { background: #e8f0fe; color: #1967d2; border: 1px solid #d2e3fc; }
.ag-badge-info    { background: #fef7e0; color: #b06000; border: 1px solid #feefc3; }
.ag-card-body { padding: 16px 18px; }
.ag-grid {
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
  gap: 10px;
  margin-bottom: 12px;
}
.ag-kv {
  background: #f8f9fa;
  border: 1px solid #e8eaed;
  border-radius: 8px;
  padding: 10px 12px;
}
.ag-kv-label {
  font-size: 11px;
  font-weight: 600;
  color: #5f6368;
  text-transform: uppercase;
  margin-bottom: 4px;
}
.ag-kv-val {
  font-size: 13px;
  font-weight: 600;
  color: #202124;
  word-break: break-all;
  font-family: 'Roboto Mono', monospace;
}
.ag-table {
  width: 100%;
  border-collapse: collapse;
  font-size: 13px;
  margin-top: 8px;
}
.ag-table th {
  text-align: left;
  padding: 8px 10px;
  background: #f1f3f4;
  color: #3c4043;
  font-weight: 600;
  border-bottom: 1px solid #dadce0;
}
.ag-table td {
  padding: 8px 10px;
  border-bottom: 1px solid #f1f3f4;
  color: #202124;
}
.ag-details {
  margin-top: 12px;
  border-top: 1px solid #e8eaed;
  padding-top: 10px;
}
.ag-details summary {
  cursor: pointer;
  font-size: 12px;
  font-weight: 600;
  color: #1a73e8;
  user-select: none;
}
.ag-pre {
  background: #1e1e2e;
  color: #cdd6f4;
  padding: 12px;
  border-radius: 8px;
  font-family: 'Roboto Mono', monospace;
  font-size: 12px;
  overflow-x: auto;
  max-height: 280px;
  margin-top: 8px;
  white-space: pre-wrap;
}
@keyframes ag-spin { 0% { transform: rotate(0deg); } 100% { transform: rotate(360deg); } }
.ag-spinner {
  display: inline-block;
  width: 14px;
  height: 14px;
  border: 2px solid #1967d2;
  border-top-color: transparent;
  border-radius: 50%;
  animation: ag-spin 0.8s linear infinite;
}
</style>
"""

def now_utc():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

def show_progress(title, subtitle="Executing command..."):
    return display(HTML(UI_CSS + f"""
    <div class="ag-card">
      <div class="ag-card-header">
        <div class="ag-card-title"><span class="ag-spinner"></span> {html.escape(title)}</div>
        <span class="ag-badge ag-badge-running">RUNNING</span>
      </div>
      <div class="ag-card-body" style="font-size:13px;color:#5f6368;">
        {html.escape(subtitle)}
      </div>
    </div>
    """), display_id=True)

def render_card(handle, title, status="SUCCESS", badge_text=None, metrics=None, table_rows=None, table_headers=None, logs="", summary_html=""):
    badge_cls = "ag-badge-success" if status == "SUCCESS" else ("ag-badge-error" if status == "ERROR" else "ag-badge-info")
    icon = "✅" if status == "SUCCESS" else ("❌" if status == "ERROR" else "ℹ️")
    b_text = badge_text or status

    metrics_html = ""
    if metrics:
        items = "".join(
            f'<div class="ag-kv"><div class="ag-kv-label">{html.escape(str(k))}</div><div class="ag-kv-val">{html.escape(str(v))}</div></div>'
            for k, v in metrics.items()
        )
        metrics_html = f'<div class="ag-grid">{items}</div>'

    table_html = ""
    if table_rows and table_headers:
        ths = "".join(f"<th>{html.escape(str(h))}</th>" for h in table_headers)
        trs = ""
        for row in table_rows:
            tds = "".join(f"<td>{c}</td>" for c in row)
            trs += f"<tr>{tds}</tr>"
        table_html = f'<table class="ag-table"><thead><tr>{ths}</tr></thead><tbody>{trs}</tbody></table>'

    details_html = ""
    if logs:
        details_html = f"""
        <div class="ag-details">
          <details>
            <summary>🔎 View Raw Execution & Debug Logs ({now_utc()})</summary>
            <pre class="ag-pre">{html.escape(logs)}</pre>
          </details>
        </div>
        """

    card_html = UI_CSS + f"""
    <div class="ag-card">
      <div class="ag-card-header">
        <div class="ag-card-title">{icon} {html.escape(title)}</div>
        <span class="ag-badge {badge_cls}">{html.escape(b_text)}</span>
      </div>
      <div class="ag-card-body">
        {summary_html}
        {metrics_html}
        {table_html}
        {details_html}
      </div>
    </div>
    """
    if handle:
        handle.update(HTML(card_html))
    else:
        display(HTML(card_html))

def run_cmd(cmd, env=None):
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)
    proc = subprocess.run(cmd, shell=True, text=True, capture_output=True, env=merged_env)
    combined = (proc.stdout or "") + ("\n" + proc.stderr if proc.stderr else "")
    return proc.returncode, (proc.stdout or "").strip(), combined.strip()

h = show_progress("Step 1: Initializing Configuration & Authenticating with Google Cloud", "Verifying workspace repository and GCP credentials...")
logs_accum = []

try:
    if "google.colab" in sys.modules:
        from google.colab import auth
        auth.authenticate_user()
        logs_accum.append("[auth] Authenticated via google.colab.auth.authenticate_user()")

    if os.path.exists("src/main.go") and os.path.exists("apigee/jdbc-policy-proxy.template.yaml"):
        REPO_ROOT = os.getcwd()
    elif os.path.exists("../src/main.go") and os.path.exists("../apigee/jdbc-policy-proxy.template.yaml"):
        REPO_ROOT = os.path.abspath("..")
    else:
        target_dir = "/content/apigee-cloudrun-jdbc-policy" if os.path.exists("/content") else os.path.abspath("./apigee-cloudrun-jdbc-policy")
        if not os.path.exists(os.path.join(target_dir, "src/main.go")):
            rc, _, out = run_cmd(f"git clone --depth 1 {REPO_URL} {target_dir}")
            logs_accum.append(f"[git clone] {out}")
        REPO_ROOT = target_dir

    if not PROJECT_ID.strip():
        _, detected_proj, _ = run_cmd("gcloud config get-value project 2>/dev/null")
        if detected_proj and detected_proj != "(unset)":
            PROJECT_ID = detected_proj.strip()

    if not PROJECT_ID.strip():
        raise ValueError("PROJECT_ID is required. Please enter your GCP Project ID in the form above and click Play.")

    rc, _, out = run_cmd(f"gcloud config set project {PROJECT_ID} --quiet")
    logs_accum.append(out)
    _, active_account, _ = run_cmd("gcloud config get-value account 2>/dev/null")

    STATE = {
        "PROJECT_ID": PROJECT_ID.strip(),
        "APIGEE_ORG": PROJECT_ID.strip(),
        "REGION": REGION.strip(),
        "SERVICE_NAME": SERVICE_NAME.strip(),
        "CUSTOM_AUDIENCE": CUSTOM_AUDIENCE.strip(),
        "APIGEE_ENV": APIGEE_ENV.strip(),
        "APIGEE_HOSTNAME": APIGEE_HOSTNAME.strip(),
        "RUNTIME_SA_NAME": "jdbc-cloudrun-runtime",
        "RUNTIME_SA_EMAIL": f"jdbc-cloudrun-runtime@{PROJECT_ID.strip()}.iam.gserviceaccount.com",
        "INVOKER_SA_NAME": "apigee-jdbc-invoker",
        "INVOKER_SA_EMAIL": f"apigee-jdbc-invoker@{PROJECT_ID.strip()}.iam.gserviceaccount.com",
        "KVM_NAME": "jdbc-datasources-kvm",
        "PROPERTYSET_NAME": "jdbc",
        "ADMIN_PROXY_NAME": "jdbc-admin-v1",
        "ADMIN_BASE_PATH": "/v1/jdbc-admin",
        "POLICY_PROXY_NAME": "jdbc-policy-callout-v1",
        "POLICY_BASE_PATH": "/v1/jdbc-policy",
        "TEST_DB_VM_NAME": "jdbc-onprem-sim-db",
        "TEST_DB_ZONE": f"{REGION.strip()}-b",
        "TEST_DB_FIREWALL_RULE": "allow-cloudrun-to-jdbc-vm",
        "REPO_ROOT": REPO_ROOT,
        "ACTIVE_ACCOUNT": active_account or "Authenticated User",
    }

    render_card(
        h,
        "Step 1: Configuration & GCP Authentication Ready",
        status="SUCCESS",
        badge_text="AUTHENTICATED",
        metrics={
            "GCP Project / Apigee Org": STATE["PROJECT_ID"],
            "Cloud Run Region": STATE["REGION"],
            "Cloud Run Service": STATE["SERVICE_NAME"],
            "Custom OIDC Audience": STATE["CUSTOM_AUDIENCE"],
            "Authenticated Account": STATE["ACTIVE_ACCOUNT"],
            "Apigee Env / Host Override": f"{STATE['APIGEE_ENV'] or '(Auto-Discover)'} / {STATE['APIGEE_HOSTNAME'] or '(Auto-Discover)'}",
        },
        logs="\n".join(logs_accum),
    )
except Exception as e:
    render_card(h, "Step 1: Configuration Error", status="ERROR", badge_text="FAILED", summary_html=f"<p style='color:#c5221f;font-weight:600;'>{html.escape(str(e))}</p>", logs="\n".join(logs_accum))

In [ ]:
#@title Step 2: Enable GCP APIs & Configure Zero-to-One IAM { display-mode: "form" }
h = show_progress("Step 2: Enabling GCP APIs & Provisioning Service Accounts", "Enabling Cloud Run, Cloud Build, Artifact Registry, IAM, and Apigee APIs...")
logs_accum = []
checklist_rows = []

try:
    if "STATE" not in globals():
        raise RuntimeError("Please run Step 1 first to initialize configuration.")

    proj = STATE["PROJECT_ID"]
    runtime_sa = STATE["RUNTIME_SA_EMAIL"]
    invoker_sa = STATE["INVOKER_SA_EMAIL"]

    apis = [
        "run.googleapis.com",
        "cloudbuild.googleapis.com",
        "artifactregistry.googleapis.com",
        "iam.googleapis.com",
        "iamcredentials.googleapis.com",
        "cloudresourcemanager.googleapis.com",
        "apigee.googleapis.com",
    ]
    rc, _, out = run_cmd(f"gcloud services enable {' '.join(apis)} --project={proj} --quiet")
    logs_accum.append(f"[services enable]\n{out}")
    if rc != 0:
        raise RuntimeError(f"Failed to enable GCP APIs: {out}")

    checklist_rows.append([
        "Google Cloud APIs",
        "<code>run</code>, <code>cloudbuild</code>, <code>artifactregistry</code>, <code>iam</code>, <code>apigee</code>",
        '<span class="ag-badge ag-badge-success">ENABLED</span>',
    ])

    _, proj_num, _ = run_cmd(f'gcloud projects describe {proj} --format="value(projectNumber)"')
    proj_num = proj_num.strip()
    compute_sa = f"{proj_num}-compute@developer.gserviceaccount.com"
    apigee_agent = f"service-{proj_num}@gcp-sa-apigee.iam.gserviceaccount.com"

    for role in ["roles/storage.admin", "roles/artifactregistry.writer", "roles/logging.logWriter", "roles/cloudbuild.builds.builder"]:
        _, _, out = run_cmd(
            f"gcloud projects add-iam-policy-binding {proj} --member=serviceAccount:{compute_sa} --role={role} --condition=None --quiet"
        )
        logs_accum.append(out)

    rc, _, _ = run_cmd(f"gcloud iam service-accounts describe {runtime_sa} --project={proj}")
    if rc != 0:
        rc, _, out = run_cmd(
            f'gcloud iam service-accounts create {STATE["RUNTIME_SA_NAME"]} '
            f'--display-name="Cloud Run JDBC Policy Connector Runtime SA" --project={proj} --quiet'
        )
        logs_accum.append(out)
        time.sleep(5)

    for role in ["roles/logging.logWriter", "roles/monitoring.metricWriter", "roles/apigee.environmentAdmin"]:
        run_cmd(f"gcloud projects add-iam-policy-binding {proj} --member=serviceAccount:{runtime_sa} --role={role} --condition=None --quiet")

    checklist_rows.append([
        "Cloud Run Runtime SA",
        f"<code>{html.escape(runtime_sa)}</code> (with KVM Auto-Sync Role)",
        '<span class="ag-badge ag-badge-success">PROVISIONED</span>',
    ])

    rc, _, _ = run_cmd(f"gcloud iam service-accounts describe {invoker_sa} --project={proj}")
    if rc != 0:
        rc, _, out = run_cmd(
            f'gcloud iam service-accounts create {STATE["INVOKER_SA_NAME"]} '
            f'--display-name="Apigee X to Cloud Run JDBC Invoker SA" --project={proj} --quiet'
        )
        logs_accum.append(out)
        time.sleep(5)

    checklist_rows.append([
        "Apigee Proxy Invoker SA",
        f"<code>{html.escape(invoker_sa)}</code>",
        '<span class="ag-badge ag-badge-success">PROVISIONED</span>',
    ])

    for role in ["roles/iam.serviceAccountTokenCreator", "roles/iam.serviceAccountUser"]:
        _, _, out = run_cmd(
            f"gcloud iam service-accounts add-iam-policy-binding {invoker_sa} "
            f"--project={proj} --member=serviceAccount:{apigee_agent} --role={role} --condition=None --quiet"
        )
        logs_accum.append(out)

    checklist_rows.append([
        "Apigee Service Agent OIDC Binding",
        f"<code>{html.escape(apigee_agent)}</code> &rarr; <code>roles/iam.serviceAccountTokenCreator</code>",
        '<span class="ag-badge ag-badge-success">BOUND</span>',
    ])

    render_card(
        h,
        "Step 2: GCP APIs & Service Accounts Provisioned",
        status="SUCCESS",
        badge_text="IAM READY",
        table_headers=["Resource Type", "Identifier / Binding", "Status"],
        table_rows=checklist_rows,
        logs="\n".join(logs_accum),
    )
except Exception as e:
    render_card(h, "Step 2: API & IAM Setup Failed", status="ERROR", badge_text="FAILED", summary_html=f"<p style='color:#c5221f;font-weight:600;'>{html.escape(str(e))}</p>", logs="\n".join(logs_accum))

In [ ]:
#@title Step 3: Deploy Private Always-Warm Cloud Run JDBC Connector { display-mode: "form" }
h = show_progress("Step 3: Building & Deploying Private Always-Warm Cloud Run Connector", "Building Go SQL/ODBC container and deploying with --min-instances=1, --no-cpu-throttling, and Custom OIDC Audience...")
logs_accum = []

try:
    if "STATE" not in globals():
        raise RuntimeError("Please run Step 1 and Step 2 first.")

    proj = STATE["PROJECT_ID"]
    region = STATE["REGION"]
    svc = STATE["SERVICE_NAME"]
    aud = STATE["CUSTOM_AUDIENCE"]
    runtime_sa = STATE["RUNTIME_SA_EMAIL"]
    invoker_sa = STATE["INVOKER_SA_EMAIL"]
    kvm = STATE["KVM_NAME"]
    src_dir = os.path.join(STATE["REPO_ROOT"], "src")

    deploy_cmd = (
        f'gcloud run deploy "{svc}" '
        f'--source="{src_dir}" '
        f'--project="{proj}" '
        f'--region="{region}" '
        f'--service-account="{runtime_sa}" '
        f'--no-allow-unauthenticated '
        f'--ingress="all" '
        f'--min-instances=1 '
        f'--max-instances=5 '
        f'--no-cpu-throttling '
        f'--memory=512Mi --cpu=1 '
        f'--set-custom-audiences="{aud}" '
        f'--set-env-vars="JDBC_DATASOURCES_CONFIG_PATH=/etc/jdbc/datasources.json,APIGEE_ORG={proj},APIGEE_ENV={STATE["APIGEE_ENV"] or "dev"},APIGEE_KVM_NAME={kvm}" '
        f'--quiet'
    )
    rc, _, out = run_cmd(deploy_cmd)
    logs_accum.append(out)
    if rc != 0:
        raise RuntimeError(f"Cloud Run deployment failed:\n{out}")

    rc, _, inv_out = run_cmd(
        f'gcloud run services add-iam-policy-binding "{svc}" '
        f'--project="{proj}" --region="{region}" '
        f'--member="serviceAccount:{invoker_sa}" --role="roles/run.invoker" --quiet'
    )
    logs_accum.append(inv_out)

    _, desc_json, _ = run_cmd(f'gcloud run services describe "{svc}" --project="{proj}" --region="{region}" --format=json')
    svc_meta = json.loads(desc_json)
    cloud_run_url = svc_meta.get("status", {}).get("url", "")
    latest_rev = svc_meta.get("status", {}).get("latestReadyRevisionName", "")
    STATE["CLOUD_RUN_URL"] = cloud_run_url
    STATE["CLOUD_RUN_REVISION"] = latest_rev

    render_card(
        h,
        "Step 3: Private Always-Warm Cloud Run JDBC Connector Deployed",
        status="SUCCESS",
        badge_text="ALWAYS-WARM POOL",
        metrics={
            "Cloud Run Service URL": cloud_run_url,
            "Active Revision": latest_rev,
            "Custom OIDC Audience": aud,
            "Keep-Alive Mode": "--min-instances=1 --no-cpu-throttling (30s Heartbeat)",
            "Runtime Service Account": runtime_sa,
            "Authorized Invoker (roles/run.invoker)": invoker_sa,
        },
        logs="\n".join(logs_accum),
    )
except Exception as e:
    render_card(h, "Step 3: Cloud Run Deployment Failed", status="ERROR", badge_text="FAILED", summary_html=f"<p style='color:#c5221f;font-weight:600;'>{html.escape(str(e))}</p>", logs="\n".join(logs_accum))

In [ ]:
#@title Step 4: Auto-Discover Apigee Environment & Hostname { display-mode: "form" }
import urllib.request

h = show_progress("Step 4: Querying Apigee Management API for Environment & Hostname", "Discovering Apigee X environments and environment group attachments...")
logs_accum = []

try:
    if "STATE" not in globals():
        raise RuntimeError("Please run Step 1 first.")

    org = STATE["APIGEE_ORG"]
    _, token, _ = run_cmd("gcloud auth print-access-token")
    if not token:
        raise RuntimeError("Unable to obtain GCP access token via gcloud auth print-access-token.")

    def apigee_get(path):
        url = f"https://apigee.googleapis.com/v1/organizations/{org}/{path}"
        req = urllib.request.Request(url, headers={"Authorization": f"Bearer {token}"})
        with urllib.request.urlopen(req, timeout=15) as resp:
            body = resp.read().decode("utf-8")
            logs_accum.append(f"GET {url} -> {resp.status}\n{body}")
            return json.loads(body)

    resolved_env = STATE["APIGEE_ENV"]
    resolved_host = STATE["APIGEE_HOSTNAME"]
    matched_envgroup = "(Manual Override)"

    if not resolved_env:
        envs_list = apigee_get("environments")
        if not envs_list:
            raise RuntimeError(f"No Apigee environments found in organization '{org}'.")
        resolved_env = envs_list[0]

    if not resolved_host:
        eg_resp = apigee_get("envgroups")
        for eg in eg_resp.get("environmentGroups", []):
            eg_name = eg.get("name")
            hostnames = eg.get("hostnames", [])
            att_resp = apigee_get(f"envgroups/{eg_name}/attachments")
            for att in att_resp.get("environmentGroupAttachments", []):
                if att.get("environment") == resolved_env and hostnames:
                    resolved_host = hostnames[0]
                    matched_envgroup = eg_name
                    break
            if resolved_host:
                break

    if not resolved_host:
        raise RuntimeError(f"Could not auto-discover hostname for environment '{resolved_env}'.")

    STATE["APIGEE_ENV"] = resolved_env
    STATE["APIGEE_HOSTNAME"] = resolved_host
    STATE["APIGEE_ENVGROUP"] = matched_envgroup

    render_card(
        h,
        "Step 4: Apigee X Environment & Ingress Routing Resolved",
        status="SUCCESS",
        badge_text="AUTO-DISCOVERED",
        metrics={
            "Apigee Organization": org,
            "Target Environment": resolved_env,
            "Environment Group": matched_envgroup,
            "Resolved Ingress Hostname": resolved_host,
            "Admin Proxy Endpoint": f"https://{resolved_host}{STATE['ADMIN_BASE_PATH']}/datasources",
            "Runtime Policy Endpoint": f"https://{resolved_host}{STATE['POLICY_BASE_PATH']}",
        },
        logs="\n".join(logs_accum),
    )
except Exception as e:
    render_card(h, "Step 4: Apigee Topology Discovery Failed", status="ERROR", badge_text="FAILED", summary_html=f"<p style='color:#c5221f;font-weight:600;'>{html.escape(str(e))}</p>", logs="\n".join(logs_accum))

In [ ]:
#@title Step 5: Deploy Encrypted KVM, jdbc.properties PropertySet & Apigee Proxies { display-mode: "form" }
h = show_progress("Step 5: Provisioning Encrypted KVM, PropertySet & Deploying Apigee Proxies via AFT", "Uploading jdbc.properties and deploying jdbc-admin-v1 & jdbc-policy-callout-v1...")
logs_accum = []

try:
    if "STATE" not in globals() or not STATE.get("CLOUD_RUN_URL") or not STATE.get("APIGEE_ENV"):
        raise RuntimeError("Please run Steps 1 through 4 first.")

    org = STATE["APIGEE_ORG"]
    env = STATE["APIGEE_ENV"]
    kvm = STATE["KVM_NAME"]
    propset = STATE["PROPERTYSET_NAME"]
    admin_proxy = STATE["ADMIN_PROXY_NAME"]
    admin_path = STATE["ADMIN_BASE_PATH"]
    policy_proxy = STATE["POLICY_PROXY_NAME"]
    policy_path = STATE["POLICY_BASE_PATH"]
    cloud_run_url = STATE["CLOUD_RUN_URL"]
    aud = STATE["CUSTOM_AUDIENCE"]
    invoker_sa = STATE["INVOKER_SA_EMAIL"]
    repo_root = STATE["REPO_ROOT"]

    _, token, _ = run_cmd("gcloud auth print-access-token")

    # 1. Create Encrypted Environment KVM
    kvm_cmd = (
        f'curl -s -X POST "https://apigee.googleapis.com/v1/organizations/{org}/environments/{env}/keyvaluemaps" '
        f'-H "Authorization: Bearer {token}" -H "Content-Type: application/json" '
        f'-d \'{{"name": "{kvm}", "encrypted": true}}\''
    )
    _, _, kvm_out = run_cmd(kvm_cmd)
    logs_accum.append(f"[KVM Create] {kvm_out}")

    # 2. Upload jdbc.properties as Environment PropertySet
    props_file = os.path.join(repo_root, "apigee", "jdbc.properties")
    _, prop_code, _ = run_cmd(
        f'curl -s -o /dev/null -w "%{{http_code}}" -H "Authorization: Bearer {token}" '
        f'"https://apigee.googleapis.com/v1/organizations/{org}/environments/{env}/resourcefiles/properties/{propset}"'
    )
    if prop_code.strip() == "200":
        _, _, prop_out = run_cmd(
            f'curl -s -X PUT "https://apigee.googleapis.com/v1/organizations/{org}/environments/{env}/resourcefiles/properties/{propset}" '
            f'-H "Authorization: Bearer {token}" -H "Content-Type: application/octet-stream" --data-binary @"{props_file}"'
        )
    else:
        _, _, prop_out = run_cmd(
            f'curl -s -X POST "https://apigee.googleapis.com/v1/organizations/{org}/environments/{env}/resourcefiles?name={propset}&type=properties" '
            f'-H "Authorization: Bearer {token}" -H "Content-Type: application/octet-stream" --data-binary @"{props_file}"'
        )
    logs_accum.append(f"[PropertySet {propset}.properties] {prop_out}")

    # 3. Render AFT YAML Templates
    build_dir = os.path.join(repo_root, "apigee", ".build")
    os.makedirs(build_dir, exist_ok=True)

    with open(os.path.join(repo_root, "apigee", "jdbc-admin-proxy.template.yaml"), "r") as f:
        admin_yaml = (
            f.read()
            .replace("__ADMIN_PROXY_NAME__", admin_proxy)
            .replace("__ADMIN_BASE_PATH__", admin_path)
            .replace("__KVM_NAME__", kvm)
            .replace("__CLOUD_RUN_URL__", cloud_run_url)
            .replace("__CUSTOM_AUDIENCE__", aud)
        )
    admin_yaml_path = os.path.join(build_dir, f"{admin_proxy}.aft.yaml")
    with open(admin_yaml_path, "w") as f:
        f.write(admin_yaml)

    with open(os.path.join(repo_root, "apigee", "jdbc-policy-proxy.template.yaml"), "r") as f:
        policy_yaml = (
            f.read()
            .replace("__POLICY_PROXY_NAME__", policy_proxy)
            .replace("__POLICY_BASE_PATH__", policy_path)
            .replace("__CLOUD_RUN_URL__", cloud_run_url)
            .replace("__CUSTOM_AUDIENCE__", aud)
        )
    policy_yaml_path = os.path.join(build_dir, f"{policy_proxy}.aft.yaml")
    with open(policy_yaml_path, "w") as f:
        f.write(policy_yaml)

    aft_bin = shutil.which("aft") or os.path.join(repo_root, "aft")
    if not os.path.exists(aft_bin):
        _, _, dl_out = run_cmd(
            f'curl -sL "https://github.com/apigee/apigee-templater/releases/latest/download/aft-linux-x64" -o "{aft_bin}" && chmod +x "{aft_bin}"'
        )
        logs_accum.append(f"[Download aft] {dl_out}")

    for p_name, y_path in [(admin_proxy, admin_yaml_path), (policy_proxy, policy_yaml_path)]:
        zip_path = os.path.join(build_dir, f"{p_name}.zip")
        _, _, out1 = run_cmd(f'"{aft_bin}" -i "{y_path}" -o "{zip_path}" --no-anim')
        if p_name == policy_proxy:
            import zipfile
            with zipfile.ZipFile(zip_path, "a") as zf:
                if "apiproxy/resources/properties/jdbc.properties" not in zf.namelist():
                    zf.write(props_file, "apiproxy/resources/properties/jdbc.properties")
            _, imp_out, _ = run_cmd(f'curl -sf -X POST "https://apigee.googleapis.com/v1/organizations/{org}/apis?action=import&name={p_name}" -H "Authorization: Bearer {token}" -F "file=@{zip_path}"')
            rev = json.loads(imp_out).get("revision", "1")
            rc2, _, out2 = run_cmd(f'curl -sf -X POST "https://apigee.googleapis.com/v1/organizations/{org}/environments/{env}/apis/{p_name}/revisions/{rev}/deployments?override=true&serviceAccount={invoker_sa}" -H "Authorization: Bearer {token}"')
        else:
            rc2, _, out2 = run_cmd(f'"{aft_bin}" -i "{y_path}" -o "{org}:{p_name}:{env}:{invoker_sa}" -t "{token}" --no-anim')
        logs_accum.append(f"[AFT {p_name}]\n{out1}\n{out2}")
        if rc2 != 0:
            raise RuntimeError(f"Failed deploying {p_name} via AFT:\n{out2}")

    table_rows = [
        [
            f"<code>{html.escape(admin_proxy)}</code>",
            f"<code>{html.escape(admin_path)}</code>",
            f"Encrypted KVM (<code>{html.escape(kvm)}</code>) + Cloud Run <code>TargetEndpoint</code>",
            f"<code>{html.escape(aud)}</code>",
            '<span class="ag-badge ag-badge-success">DEPLOYED</span>',
        ],
        [
            f"<code>{html.escape(policy_proxy)}</code>",
            f"<code>{html.escape(policy_path)}</code>",
            f"PropertySet (<code>{html.escape(propset)}.properties</code>) + Single <code>ServiceCallout</code>",
            f"<code>{html.escape(aud)}</code>",
            '<span class="ag-badge ag-badge-success">DEPLOYED</span>',
        ],
    ]

    render_card(
        h,
        "Step 5: Apigee X KVM, PropertySet & Proxies Deployed",
        status="SUCCESS",
        badge_text="LIVE IN APIGEE X",
        metrics={
            "Apigee Environment": env,
            "Encrypted Environment KVM": f"{kvm} (encrypted=true)",
            "Environment PropertySet": f"{propset}.properties",
            "Attached Proxy Identity": invoker_sa,
        },
        table_headers=["Proxy Name", "Base Path", "Architecture Pattern", "OIDC Audience", "Status"],
        table_rows=table_rows,
        logs="\n".join(logs_accum),
    )
except Exception as e:
    render_card(h, "Step 5: Apigee Deployment Failed", status="ERROR", badge_text="FAILED", summary_html=f"<p style='color:#c5221f;font-weight:600;'>{html.escape(str(e))}</p>", logs="\n".join(logs_accum))

In [ ]:
#@title Step 6: Interactive End-to-End Verification Suite (Built-In + Public PostgreSQL) { display-mode: "form" }
SCENARIO_FILTER = "Run All 9 Verification Checks (IAM Gate + Admin + Runtime SQL)" #@param ["Run All 9 Verification Checks (IAM Gate + Admin + Runtime SQL)", "1. Direct Unauthenticated Cloud Run Call (IAM Gate Rejection)", "2. Step 1 (Admin): Onboard Live Public PostgreSQL (hh-pgsql-public.ebi.ac.uk)", "3. Step 2 (Runtime): SELECT & DML Transactions via jdbc.properties", "4. Step 2 (Runtime): Live External PostgreSQL Prepared Query"]

import urllib.request
import urllib.error
import ssl

h = show_progress("Step 6: Executing End-to-End JDBC/ODBC Verification Suite", "Running live HTTP verification checks against Cloud Run and Apigee X...")

try:
    if "STATE" not in globals() or not STATE.get("APIGEE_HOSTNAME") or not STATE.get("CLOUD_RUN_URL"):
        raise RuntimeError("Please run Steps 1 through 5 before executing the verification suite.")

    host = STATE["APIGEE_HOSTNAME"]
    cloud_run_url = STATE["CLOUD_RUN_URL"]
    aud = STATE["CUSTOM_AUDIENCE"]
    admin_url = f"https://{host}{STATE['ADMIN_BASE_PATH']}/datasources"
    policy_url = f"https://{host}{STATE['POLICY_BASE_PATH']}"

    ssl_ctx = ssl.create_default_context()
    ssl_ctx.check_hostname = False
    ssl_ctx.verify_mode = ssl.CERT_NONE

    def exec_http(name, method, url, expected_codes, json_body=None, aud_verified=True):
        req_headers = {"Accept": "application/json"}
        data_bytes = None
        if json_body is not None:
            data_bytes = json.dumps(json_body).encode("utf-8")
            req_headers["Content-Type"] = "application/json"

        t0 = time.perf_counter()
        status_code = 0
        resp_text = ""
        try:
            req = urllib.request.Request(url, data=data_bytes, headers=req_headers, method=method)
            with urllib.request.urlopen(req, context=ssl_ctx, timeout=20) as resp:
                status_code = resp.status
                resp_text = resp.read().decode("utf-8", errors="replace")
        except urllib.error.HTTPError as he:
            status_code = he.code
            resp_text = he.read().decode("utf-8", errors="replace")
        except Exception as ex:
            status_code = 599
            resp_text = str(ex)
        latency_ms = int((time.perf_counter() - t0) * 1000)

        try:
            parsed = json.loads(resp_text)
            pretty_payload = json.dumps(parsed, indent=2)
        except Exception:
            pretty_payload = resp_text[:1200]

        passed = status_code in expected_codes
        return {
            "name": name,
            "method": method,
            "url": url,
            "status_code": status_code,
            "passed": passed,
            "latency_ms": latency_ms,
            "aud_verified": aud if aud_verified else "None (Blocked by Cloud Run IAM Gate)",
            "payload": pretty_payload,
        }

    tests_to_run = []

    if SCENARIO_FILTER in ["Run All 9 Verification Checks (IAM Gate + Admin + Runtime SQL)", "1. Direct Unauthenticated Cloud Run Call (IAM Gate Rejection)"]:
        tests_to_run.append(lambda: exec_http(
            "Test 0: Direct Unauthenticated Call to Cloud Run URL (IAM Gate Check -> 403)",
            "GET",
            f"{cloud_run_url}/v1/jdbc/datasources",
            expected_codes=[401, 403],
            aud_verified=False,
        ))

    if SCENARIO_FILTER in ["Run All 9 Verification Checks (IAM Gate + Admin + Runtime SQL)", "2. Step 1 (Admin): Onboard Live Public PostgreSQL (hh-pgsql-public.ebi.ac.uk)"]:
        tests_to_run.append(lambda: exec_http(
            "Test 1: List Initial Warm Datasource Pools (enterprise-sqlite)",
            "GET",
            admin_url,
            expected_codes=[200],
        ))
        tests_to_run.append(lambda: exec_http(
            "Test 2: Onboard Public PostgreSQL (hh-pgsql-public.ebi.ac.uk:5432) -> Encrypted KVM + 3 Warm Sockets",
            "POST",
            admin_url,
            expected_codes=[200],
            json_body={
                "datasource_id": "pg-rna-public",
                "description": "Public EBI RNAcentral PostgreSQL Service - Persistent Warm Pool",
                "driver": "postgres",
                "jdbc_url": "jdbc:postgresql://hh-pgsql-public.ebi.ac.uk:5432/pfmegrnargs",
                "username": "reader",
                "password": "NWDMCE5xdipIjRrp",
                "min_pool_size": 3,
                "max_pool_size": 10,
                "keep_alive_sec": 30,
                "query_timeout_ms": 10000,
            },
        ))
        tests_to_run.append(lambda: exec_http(
            "Test 3: Verify 'pg-rna-public' Warm Pool Telemetry & Masked Password",
            "GET",
            f"{admin_url}?id=pg-rna-public",
            expected_codes=[200],
        ))
        tests_to_run.append(lambda: exec_http(
            "Test 4: Admin Conditional Guard Rejects Missing datasource_id (-> 400 Bad Request)",
            "POST",
            admin_url,
            expected_codes=[400],
            json_body={"driver": "postgres", "jdbc_url": "jdbc:postgresql://localhost:5432/db"},
        ))

    if SCENARIO_FILTER in ["Run All 9 Verification Checks (IAM Gate + Admin + Runtime SQL)", "3. Step 2 (Runtime): SELECT & DML Transactions via jdbc.properties"]:
        tests_to_run.append(lambda: exec_http(
            "Test 5: SELECT Subscriber (?id=1001) via {propertyset.jdbc.datasource_id} (-> 200 OK)",
            "GET",
            f"{policy_url}/subscribers?id=1001",
            expected_codes=[200],
        ))
        tests_to_run.append(lambda: exec_http(
            "Test 6: SELECT Non-Existent Subscriber (?id=9999) with fail_if_empty=true (-> 404 Not Found)",
            "GET",
            f"{policy_url}/subscribers?id=9999",
            expected_codes=[404],
        ))
        tests_to_run.append(lambda: exec_http(
            "Test 7: Parameterized INSERT Order DML with Transaction (POST /orders -> 200 OK)",
            "POST",
            f"{policy_url}/orders",
            expected_codes=[200],
            json_body={
                "subscriber_id": 1001,
                "sku": "DEDICATED-CLOUD-INTERCONNECT-10G",
                "amount": 3150.00,
                "currency": "EUR",
            },
        ))

    if SCENARIO_FILTER in ["Run All 9 Verification Checks (IAM Gate + Admin + Runtime SQL)", "4. Step 2 (Runtime): Live External PostgreSQL Prepared Query"]:
        tests_to_run.append(lambda: exec_http(
            "Test 8: Single-Row SELECT on Live External PostgreSQL (?upi=URS0000000001) via {propertyset.jdbc.external_datasource_id} (-> 200 OK)",
            "GET",
            f"{policy_url}/rna?upi=URS0000000001",
            expected_codes=[200],
        ))
        tests_to_run.append(lambda: exec_http(
            "Test 9: Multi-Row / Large SELECT on Live External PostgreSQL (?rna_type=rRNA) with max_rows Cap, rows[] Array & truncated=true (-> 200 OK)",
            "GET",
            f"{policy_url}/rna-search?rna_type=rRNA",
            expected_codes=[200],
        ))

    results = [fn() for fn in tests_to_run]
    passed_count = sum(1 for r in results if r["passed"])
    total_count = len(results)
    overall_ok = passed_count == total_count

    cards_html = []
    for r in results:
        b_cls = "ag-badge-success" if r["passed"] else "ag-badge-error"
        status_label = f"HTTP {r['status_code']} ({'PASS' if r['passed'] else 'FAIL'})"
        cards_html.append(f"""
        <div class="ag-card" style="margin:10px 0;">
          <div class="ag-card-header">
            <div class="ag-card-title">{'✅' if r['passed'] else '❌'} {html.escape(r['name'])}</div>
            <span class="ag-badge {b_cls}">{html.escape(status_label)}</span>
          </div>
          <div class="ag-card-body">
            <div class="ag-grid">
              <div class="ag-kv"><div class="ag-kv-label">Request</div><div class="ag-kv-val">{html.escape(r['method'])} {html.escape(r['url'])}</div></div>
              <div class="ag-kv"><div class="ag-kv-label">Response Latency</div><div class="ag-kv-val">{r['latency_ms']} ms</div></div>
              <div class="ag-kv"><div class="ag-kv-label">Custom OIDC Audience</div><div class="ag-kv-val">{html.escape(r['aud_verified'])}</div></div>
            </div>
            <div class="ag-details">
              <details open>
                <summary>📄 Response Payload</summary>
                <pre class="ag-pre">{html.escape(r['payload'])}</pre>
              </details>
            </div>
          </div>
        </div>
        """)

    dashboard_html = UI_CSS + f"""
    <div class="ag-card">
      <div class="ag-card-header">
        <div class="ag-card-title">{'🎉' if overall_ok else '⚠️'} End-to-End Verification Dashboard ({passed_count}/{total_count} Passed)</div>
        <span class="ag-badge {'ag-badge-success' if overall_ok else 'ag-badge-error'}">{passed_count}/{total_count} PASSED</span>
      </div>
      <div class="ag-card-body">
        <div class="ag-grid">
          <div class="ag-kv"><div class="ag-kv-label">Apigee Ingress Host</div><div class="ag-kv-val">{html.escape(host)}</div></div>
          <div class="ag-kv"><div class="ag-kv-label">Private Cloud Run Target</div><div class="ag-kv-val">{html.escape(cloud_run_url)}</div></div>
          <div class="ag-kv"><div class="ag-kv-label">Verified OIDC Audience</div><div class="ag-kv-val">{html.escape(aud)}</div></div>
        </div>
        {''.join(cards_html)}
      </div>
    </div>
    """
    h.update(HTML(dashboard_html))
except Exception as e:
    render_card(h, "Step 6: Verification Suite Error", status="ERROR", badge_text="FAILED", summary_html=f"<p style='color:#c5221f;font-weight:600;'>{html.escape(str(e))}</p>")

In [ ]:
#@title Step 7 (Optional): Provision, Test, or Delete Private GCE PostgreSQL Container VM { display-mode: "form" }
VM_ACTION = "1. Create Private PostgreSQL VM + Enable Direct VPC Egress + Test via Apigee" #@param ["1. Create Private PostgreSQL VM + Enable Direct VPC Egress + Test via Apigee", "2. Delete Private PostgreSQL VM, Firewall Rule & Datasource"]
VPC_NETWORK = "default" #@param {type:"string"}
VPC_SUBNET = "default" #@param {type:"string"}

h = show_progress(
    "Step 7 (Optional): Managing Private Compute Engine PostgreSQL Container VM",
    "Executing scripts/04-optional-test-db-vm.sh..."
)

try:
    if "STATE" not in globals() or not STATE.get("APIGEE_HOSTNAME"):
        raise RuntimeError("Please run Steps 1 through 5 first.")

    script_path = os.path.join(STATE["REPO_ROOT"], "scripts", "04-optional-test-db-vm.sh")
    action_flag = "--create" if VM_ACTION.startswith("1.") else "--delete"
    cmd = (
        f'bash "{script_path}" {action_flag} '
        f'--project "{STATE["PROJECT_ID"]}" '
        f'--region "{STATE["REGION"]}" '
        f'--env "{STATE["APIGEE_ENV"]}" '
        f'--hostname "{STATE["APIGEE_HOSTNAME"]}" '
        f'--network "{VPC_NETWORK.strip()}" '
        f'--subnet "{VPC_SUBNET.strip()}"'
    )
    rc, out, full_logs = run_cmd(cmd)
    if rc != 0:
        raise RuntimeError(f"Script exited with code {rc}:\n{full_logs}")

    if action_flag == "--create":
        render_card(
            h,
            "Step 7: Private PostgreSQL Container VM Created & Verified over Direct VPC Egress",
            status="SUCCESS",
            badge_text="PRIVATE VPC VERIFIED",
            metrics={
                "VM Name": STATE["TEST_DB_VM_NAME"],
                "Zone": STATE["TEST_DB_ZONE"],
                "VPC Network / Subnet": f"{VPC_NETWORK} / {VPC_SUBNET}",
                "Egress Mode": "Cloud Run Direct VPC Egress (private-ranges-only)",
                "Datasource ID": "private-vm-postgres",
            },
            summary_html=f"<pre class='ag-pre'>{html.escape(out[-1500:])}</pre>",
            logs=full_logs,
        )
    else:
        render_card(
            h,
            "Step 7: Private PostgreSQL Container VM & Firewall Rule Deleted",
            status="SUCCESS",
            badge_text="CLEANED UP",
            metrics={
                "Deleted VM": STATE["TEST_DB_VM_NAME"],
                "Deleted Firewall Rule": STATE["TEST_DB_FIREWALL_RULE"],
                "Removed Datasource": "private-vm-postgres",
            },
            logs=full_logs,
        )
except Exception as e:
    render_card(h, "Step 7: Optional Test VM Operation Failed", status="ERROR", badge_text="FAILED", summary_html=f"<p style='color:#c5221f;font-weight:600;'>{html.escape(str(e))}</p>")